============================================
#            RIFT-EEG ANALYSIS             
============================================

Author: Christos Dalamarinis

Date: March - 2026

## Notebook 01 — Data Import & Trigger Verification

Goal: Load raw data, verify everything looks right before touching it.

Contents:
1. Cell 1 -> Imports necessary packages for all analysis
2. Cell 2 -> Configuration of data for all particpants (auto-detects subjects from folder)


In [1]:
#! Cell 1 - Importing libraries and checking versions

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

mne.set_log_level('WARNING') # Suppress MNE info messages

print(f'MNE version:    {mne.__version__}')
print(f'NumPy version:  {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Matplotlib version: {plt.matplotlib.__version__}')

MNE version:    1.11.0
NumPy version:  2.2.6
Pandas version: 2.3.3
Matplotlib version: 3.10.8


In [2]:
#! Cell 2 - Configurations and parameters for experiment 

import glob # For file searching 

#! ── Paths ────────────────────────────────────────────────────
DATA_DIR   = r'C:\Users\dalam\OneDrive\Vrije Universiteit Amsterdam\Neuropsychology\Thesis\Coding - Thesis\WM data'   
OUTPUT_DIR = r'C:\Users\dalam\OneDrive\Vrije Universiteit Amsterdam\Neuropsychology\Thesis\Coding - Thesis\WM data\derivatives'

os.makedirs(OUTPUT_DIR, exist_ok=True)

#! ── Auto-detect all .bdf files in folder ─────────────────────
bdf_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.bdf')))

if len(bdf_files) == 0:
    raise FileNotFoundError(f'No .bdf files found in {DATA_DIR}')

print('Found .bdf files:')
for i, f in enumerate(bdf_files):
    print(f'  [{i}]  {os.path.basename(f)}')

#! ── Select subject ────────────────────────────────────────────
subjects = []
for i, f in enumerate(bdf_files):
    subjects.append({
        'subject_id':   f'sub-{i+1:02d}',
        'bdf_filename': os.path.basename(f),
        'bdf_path':     f,
    })


#Old code:
# Change this index to switch subjects: 0 = first file, 1 = second, etc.
#SUBJECT_IDX  = 1

#BDF_FILENAME = os.path.basename(bdf_files[SUBJECT_IDX])
#SUBJECT_ID   = f'sub-{SUBJECT_IDX + 1:02d}'   # sub-01, sub-02, etc.

#! ── Channel names (verify in Cell 4) ─────────────────────────
MASTOID_CHANNELS = ['EXG5', 'EXG6']
EOG_CHANNELS     = ['EXG1', 'EXG2', 'EXG3', 'EXG4']
UNUSED_CHANNELS     = ['EXG7', 'EXG8']  # update label once confirmed

#! ── Experiment parameters ─────────────────────────────────────
TRIG_TRIAL_START = 20
TRIG_MEMORY_ON   = 30
TRIG_RIFT_ON     = 40
TRIG_RIFT_OFF    = 50
TRIG_BLOCK_START = 250
TRIG_BLOCK_END   = 251

NOMINAL_FLICKER_HZ = 60.0
NOMINAL_RIFT_DUR   = 2.0
N_BLOCKS           = 12
N_TRIALS_PER_BLOCK = 50
N_TRIALS_EXPECTED  = N_BLOCKS * N_TRIALS_PER_BLOCK  # 600


#! ── Print summary of selected subject and parameters ──────────
print(f'\nAll subjects queued for analysis:')
for s in subjects:
    print(f"  {s['subject_id']}  →  {s['bdf_filename']}")
print(f'\nOutput dir     : {OUTPUT_DIR}')
print(f'Expected trials: {N_TRIALS_EXPECTED}')

#Old code:
#print(f'\nSelected subject : {SUBJECT_ID}')
#print(f'BDF file         : {BDF_FILENAME}')
#print(f'Output dir       : {OUTPUT_DIR}')
#print(f'Expected trials  : {N_TRIALS_EXPECTED}')

Found .bdf files:
  [0]  sub-01.bdf
  [1]  sub-02.bdf

All subjects queued for analysis:
  sub-01  →  sub-01.bdf
  sub-02  →  sub-02.bdf

Output dir     : C:\Users\dalam\OneDrive\Vrije Universiteit Amsterdam\Neuropsychology\Thesis\Coding - Thesis\WM data\derivatives
Expected trials: 600


In [3]:
#! Cell 3 - Load raw data and print basic info
raw_dict = {}  # stores raw objects for all subjects, used by later cells

for s in subjects:
    print(f"\n{'='*50}")
    print(f"  Loading {s['subject_id']}  —  {s['bdf_filename']}")
    print(f"{'='*50}")

    raw = mne.io.read_raw_bdf(s['bdf_path'], preload=False, verbose=False)

    print(f"  Sampling rate      : {raw.info['sfreq']} Hz")
    print(f"  Number of channels : {len(raw.ch_names)}")
    print(f"  Duration           : {raw.times[-1]:.1f} s  ({raw.times[-1]/60:.1f} min)")

    raw_dict[s['subject_id']] = raw

# Old code:
#bdf_path = os.path.join(DATA_DIR, BDF_FILENAME)

#raw = mne.io.read_raw_bdf(bdf_path, preload=False, verbose=True)

#print('\n' + '='*50)
#print(f'  Sampling rate      : {raw.info["sfreq"]} Hz')
#print(f'  Number of channels : {len(raw.ch_names)}')
#print(f'  Duration           : {raw.times[-1]:.1f} s  ({raw.times[-1]/60:.1f} min)')
#print('='*50)


  Loading sub-01  —  sub-01.bdf
  Sampling rate      : 1024.0 Hz
  Number of channels : 73
  Duration           : 3335.0 s  (55.6 min)

  Loading sub-02  —  sub-02.bdf
  Sampling rate      : 1024.0 Hz
  Number of channels : 73
  Duration           : 3745.0 s  (62.4 min)


In [4]:
# Cell 4 - Print all channel names to verify MASTOID and EOG channels
#! Only needs to run once — channel layout is the same for all subjects
# We use the first subject to verify
first_raw = raw_dict[subjects[0]['subject_id']]

print('Channel names (verify MASTOID_CHANNELS and EOG_CHANNELS match these):')
print('-' * 40)
for i, ch in enumerate(first_raw.ch_names):
    print(f'  [{i:3d}]  {ch}')

Channel names (verify MASTOID_CHANNELS and EOG_CHANNELS match these):
----------------------------------------
  [  0]  Fp1
  [  1]  AF7
  [  2]  AF3
  [  3]  F1
  [  4]  F3
  [  5]  F5
  [  6]  F7
  [  7]  FT7
  [  8]  FC5
  [  9]  FC3
  [ 10]  FC1
  [ 11]  C1
  [ 12]  C3
  [ 13]  C5
  [ 14]  T7
  [ 15]  TP7
  [ 16]  CP5
  [ 17]  CP3
  [ 18]  CP1
  [ 19]  P1
  [ 20]  P3
  [ 21]  P5
  [ 22]  P7
  [ 23]  P9
  [ 24]  PO7
  [ 25]  PO3
  [ 26]  O1
  [ 27]  Iz
  [ 28]  Oz
  [ 29]  POz
  [ 30]  Pz
  [ 31]  CPz
  [ 32]  Fpz
  [ 33]  Fp2
  [ 34]  AF8
  [ 35]  AF4
  [ 36]  AFz
  [ 37]  Fz
  [ 38]  F2
  [ 39]  F4
  [ 40]  F6
  [ 41]  F8
  [ 42]  FT8
  [ 43]  FC6
  [ 44]  FC4
  [ 45]  FC2
  [ 46]  FCz
  [ 47]  Cz
  [ 48]  C2
  [ 49]  C4
  [ 50]  C6
  [ 51]  T8
  [ 52]  TP8
  [ 53]  CP6
  [ 54]  CP4
  [ 55]  CP2
  [ 56]  P2
  [ 57]  P4
  [ 58]  P6
  [ 59]  P8
  [ 60]  P10
  [ 61]  PO8
  [ 62]  PO4
  [ 63]  O2
  [ 64]  EXG1
  [ 65]  EXG2
  [ 66]  EXG3
  [ 67]  EXG4
  [ 68]  EXG5
  [ 69]  EXG6
  [ 7

In [5]:
# Cell 5 - Set channel types and print summary of channel types

for s in subjects:
    raw = raw_dict[s['subject_id']]

    ch_type_mapping = {ch: 'eog'  for ch in EOG_CHANNELS}
    ch_type_mapping.update({ch: 'misc' for ch in UNUSED_CHANNELS})  # floating, exclude these

    #! Mastoids stay as 'eeg' so MNE can use them for re-referencing in Notebook 02
    # They will be dropped after re-referencing

    raw.set_channel_types(ch_type_mapping)

    ch_types_df = pd.DataFrame({
        'channel': raw.ch_names,
        'type':    [mne.channel_type(raw.info, i) for i in range(len(raw.ch_names))]
    })

    print(f"\n{s['subject_id']} — channel type summary:")
    print(ch_types_df.groupby('type')['channel'].apply(list).to_string())


#! Quick check — confirm EXG5 and EXG6 are typed as eeg 
for s in subjects:
    raw = raw_dict[s['subject_id']]
    print(f"{s['subject_id']}:")
    for ch in MASTOID_CHANNELS:
        print(f"  {ch} → {mne.channel_type(raw.info, raw.ch_names.index(ch))}")


sub-01 — channel type summary:
type
eeg     [Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3,...
eog                              [EXG1, EXG2, EXG3, EXG4]
misc                                         [EXG7, EXG8]
stim                                             [Status]

sub-02 — channel type summary:
type
eeg     [Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3,...
eog                              [EXG1, EXG2, EXG3, EXG4]
misc                                         [EXG7, EXG8]
stim                                             [Status]
sub-01:
  EXG5 → eeg
  EXG6 → eeg
sub-02:
  EXG5 → eeg
  EXG6 → eeg


C:\Users\dalam\AppData\Local\Temp\ipykernel_4232\1563693782.py:12: RuntimeWarning: The unit for channel(s) EXG7, EXG8 has changed from V to NA.
  raw.set_channel_types(ch_type_mapping)
C:\Users\dalam\AppData\Local\Temp\ipykernel_4232\1563693782.py:12: RuntimeWarning: The unit for channel(s) EXG7, EXG8 has changed from V to NA.
  raw.set_channel_types(ch_type_mapping)


In [6]:
# Cell 6 - Extract events and verify trigger codes and counts
events_dict = {}

for s in subjects:
    raw = raw_dict[s['subject_id']]

    events = mne.find_events(
        raw,
        stim_channel='Status',
        shortest_event=1,
        mask=255,
        verbose=False
    )

    #! Remove Biosemi status artifacts (254 = hardware status bit, not a real trigger)
    events = events[~np.isin(events[:, 2], [254])]

    events_dict[s['subject_id']] = events

    print(f"\n{s['subject_id']} — trigger verification:")
    print(f"  Total events found  : {len(events)}")
    print(f"  Unique codes        : {np.unique(events[:, 2])}")

    trigger_info = [
        (TRIG_TRIAL_START, 'Trial start  (20)', N_TRIALS_EXPECTED),
        (TRIG_MEMORY_ON,   'Memory ON    (30)', N_TRIALS_EXPECTED),
        (TRIG_RIFT_ON,     'RIFT ON      (40)', N_TRIALS_EXPECTED),
        (TRIG_RIFT_OFF,    'RIFT OFF     (50)', N_TRIALS_EXPECTED),
        (TRIG_BLOCK_START, 'Block start (250)', N_BLOCKS * 2),
        (TRIG_BLOCK_END,   'Block end   (251)', N_BLOCKS),
    ]

    rows = []
    for code, label, expected in trigger_info:
        found  = int(np.sum(events[:, 2] == code))
        status = '✓' if found == expected else f'⚠  expected {expected}'
        rows.append({'Trigger': label, 'Found': found, 'Expected': expected, 'Status': status})

    print(pd.DataFrame(rows).to_string(index=False))

    # Contextual notes
    rift_on_count = int(np.sum(events[:, 2] == TRIG_RIFT_ON))
    if rift_on_count < N_TRIALS_EXPECTED:
        print(f'\n  ⚠ sub-01 note: experiment likely cut short — {N_TRIALS_EXPECTED - rift_on_count} trials missing')
    if rift_on_count > N_TRIALS_EXPECTED:
        print(f'\n  Note: counts above {N_TRIALS_EXPECTED} include practice (15) + awareness (30) trials')
        print(f'  These will be separated in Notebook 04')
    print(f'  Note: trigger 251 (block end) unreliable via serial port — not used in analysis')



#! Old Code:
#events_dict = {}  # stores events arrays for all subjects

#for s in subjects:
 #   raw = raw_dict[s['subject_id']]

  #  events = mne.find_events(
   #     raw,
    #    stim_channel='Status',
     #  verbose=False)

    #events_dict[s['subject_id']] = events

    #print(f"\n{s['subject_id']} — trigger verification:")
    #print(f"  Total events found  : {len(events)}")
    #print(f"  Unique codes        : {np.unique(events[:, 2])}")

    #trigger_info = [
     #   (TRIG_TRIAL_START, 'Trial start  (20)', N_TRIALS_EXPECTED),
      #  (TRIG_MEMORY_ON,   'Memory ON    (30)', N_TRIALS_EXPECTED),
       # (TRIG_RIFT_ON,     'RIFT ON      (40)', N_TRIALS_EXPECTED),
        #(TRIG_RIFT_OFF,    'RIFT OFF     (50)', N_TRIALS_EXPECTED),
        #(TRIG_BLOCK_START, 'Block start (250)', N_BLOCKS * 2),
        #(TRIG_BLOCK_END,   'Block end   (251)', N_BLOCKS),]

    #rows = []
    #for code, label, expected in trigger_info:
     #   found  = int(np.sum(events[:, 2] == code))
      #  status = '✓' if found == expected else f'⚠  expected {expected}'
       # rows.append({'Trigger': label, 'Found': found, 'Expected': expected, 'Status': status})

    #print(pd.DataFrame(rows).to_string(index=False))


sub-01 — trigger verification:
  Total events found  : 2369
  Unique codes        : [ 20  30  40  50 250]
          Trigger  Found  Expected          Status
Trial start  (20)    584       600 ⚠  expected 600
Memory ON    (30)    592       600 ⚠  expected 600
RIFT ON      (40)    593       600 ⚠  expected 600
RIFT OFF     (50)    593       600 ⚠  expected 600
Block start (250)      7        24  ⚠  expected 24
Block end   (251)      0        12  ⚠  expected 12

  ⚠ sub-01 note: experiment likely cut short — 7 trials missing
  Note: trigger 251 (block end) unreliable via serial port — not used in analysis

sub-02 — trigger verification:
  Total events found  : 2626
  Unique codes        : [ 20  30  40  50 250]
          Trigger  Found  Expected          Status
Trial start  (20)    637       600 ⚠  expected 600
Memory ON    (30)    654       600 ⚠  expected 600
RIFT ON      (40)    654       600 ⚠  expected 600
RIFT OFF     (50)    652       600 ⚠  expected 600
Block start (250)     29   

In [ ]:
# Cell 6b - Clean events by removing junk triggers from recording crashes
#! This cell finds and removes junk triggers from recording crashes
# It works by finding the largest time gap between consecutive trigger 40s
# Everything before that gap is discarded as junk

events_dict_clean = {}  # will replace events_dict after cleaning

for s in subjects:
    raw    = raw_dict[s['subject_id']]
    events = events_dict[s['subject_id']]
    sfreq  = raw.info['sfreq']

    rift_on_events    = events[events[:, 2] == TRIG_RIFT_ON]
    rift_on_times     = rift_on_events[:, 0] / sfreq  # convert to seconds
    rift_on_count     = len(rift_on_times)

    print(f"\n{s['subject_id']}:")
    print(f"  RIFT ON triggers before cleaning : {rift_on_count}")

    if rift_on_count == N_TRIALS_EXPECTED:
        # Already correct — no cleaning needed
        events_dict_clean[s['subject_id']] = events
        print(f"  ✓ Count matches expected ({N_TRIALS_EXPECTED}) — no cleaning needed")
        continue

    # Find gaps between consecutive trigger 40s
    gaps        = np.diff(rift_on_times)  # time between each consecutive pair
    max_gap_idx = np.argmax(gaps)         # index of the largest gap
    max_gap_s   = gaps[max_gap_idx]       # size of the largest gap in seconds
    cut_time_s  = rift_on_times[max_gap_idx + 1]  # timestamp where real session starts

    print(f"  Largest gap found    : {max_gap_s:.1f} s  (between trigger {max_gap_idx} and {max_gap_idx + 1})")
    print(f"  Real session starts  : {cut_time_s:.1f} s into recording ({cut_time_s/60:.1f} min)")
    print(f"  Junk triggers before : {max_gap_idx + 1} RIFT ON triggers discarded")

    # Keep only events at or after the cut point
    #cut_sample        = cut_time_s * sfreq   
    cut_sample = int(cut_time_s * sfreq) #! Convert to (int) for indexing!
    events_clean      = events[events[:, 0] >= cut_sample]
    events_dict_clean[s['subject_id']] = events_clean

    # Verify
    rift_on_clean = np.sum(events_clean[:, 2] == TRIG_RIFT_ON)
    status        = '✓' if rift_on_clean == N_TRIALS_EXPECTED else f'⚠ expected {N_TRIALS_EXPECTED}'
    print(f"  RIFT ON triggers after cleaning  : {rift_on_clean}  {status}")

# Replace events_dict with cleaned version for all downstream cells
events_dict = events_dict_clean

print(f"\n{'='*50}")
print(f"  Cleaning complete — events_dict updated")
print(f"  All downstream cells will use cleaned events")
print(f"{'='*50}")


sub-01:
  RIFT ON triggers before cleaning : 593
  Largest gap found    : 14.9 s  (between trigger 7 and 8)
  Real session starts  : 58.2 s into recording (1.0 min)
  Junk triggers before : 8 RIFT ON triggers discarded
  RIFT ON triggers after cleaning  : 585  ⚠ expected 600

sub-02:
  RIFT ON triggers before cleaning : 654
  Largest gap found    : 96.6 s  (between trigger 53 and 54)
  Real session starts  : 388.1 s into recording (6.5 min)
  Junk triggers before : 54 RIFT ON triggers discarded
  RIFT ON triggers after cleaning  : 600  ✓

  Cleaning complete — events_dict updated
  All downstream cells will use cleaned events


In [ ]:
# Cell 7 - Estimate flicker frequency from RIFT ON/OFF durations

freq_info_all = {}  # stores frequency estimates for all subjects

for s in subjects:
    raw    = raw_dict[s['subject_id']]
    events = events_dict[s['subject_id']]
    sfreq  = raw.info['sfreq']

    rift_on_samples  = events[events[:, 2] == TRIG_RIFT_ON,  0]
    rift_off_samples = events[events[:, 2] == TRIG_RIFT_OFF, 0]

    print(f"\n{s['subject_id']} — flicker frequency estimation:")

    if len(rift_on_samples) != len(rift_off_samples):
        print('  ⚠ WARNING: RIFT ON and RIFT OFF counts do not match!')
        continue

    rift_durations_s = (rift_off_samples - rift_on_samples) / sfreq
    mean_dur         = np.mean(rift_durations_s)
    std_dur          = np.std(rift_durations_s)
    nominal_n_frames = round(NOMINAL_RIFT_DUR * NOMINAL_FLICKER_HZ)  # 120 frames
    empirical_freq   = nominal_n_frames / mean_dur

    print(f'  RIFT duration — mean : {mean_dur*1000:.2f} ms  (expected: {NOMINAL_RIFT_DUR*1000:.0f} ms)')
    print(f'  RIFT duration — std  : {std_dur*1000:.2f} ms')
    print(f'  RIFT duration — min  : {np.min(rift_durations_s)*1000:.2f} ms')
    print(f'  RIFT duration — max  : {np.max(rift_durations_s)*1000:.2f} ms')
    print(f'  Nominal frames       : {nominal_n_frames}')
    print(f'  Empirical freq       : {empirical_freq:.4f} Hz  (nominal: {NOMINAL_FLICKER_HZ} Hz)')
    print(f'  Deviation            : {abs(empirical_freq - NOMINAL_FLICKER_HZ)*1000:.2f} mHz')

    freq_info_all[s['subject_id']] = {
        'subject_id':            s['subject_id'],
        'sfreq':                 sfreq,
        'nominal_flicker_hz':    NOMINAL_FLICKER_HZ,
        'empirical_flicker_hz':  empirical_freq,
        'mean_rift_duration_s':  mean_dur,
        'std_rift_duration_s':   std_dur,
        'n_rift_trials':         len(rift_durations_s),
        'rift_durations_s':      rift_durations_s,  # kept for plotting in next cell
    }

In [ ]:
# Cell 8 - Plot RIFT duration distributions and check for drift over session

for s in subjects:
    if s['subject_id'] not in freq_info_all:
        continue

    info             = freq_info_all[s['subject_id']]
    rift_durations_s = info['rift_durations_s']
    mean_dur         = info['mean_rift_duration_s']
    empirical_freq   = info['empirical_flicker_hz']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{s['subject_id']} — RIFT Window Duration Check", fontsize=13, fontweight='bold')

    # Histogram
    ax = axes[0]
    ax.hist(rift_durations_s * 1000, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.axvline(NOMINAL_RIFT_DUR * 1000, color='red',   linestyle='--', linewidth=1.5, label=f'Nominal ({NOMINAL_RIFT_DUR*1000:.0f} ms)')
    ax.axvline(mean_dur * 1000,         color='orange', linestyle='-',  linewidth=1.5, label=f'Mean ({mean_dur*1000:.1f} ms)')
    ax.set_xlabel('Duration (ms)')
    ax.set_ylabel('Trial count')
    ax.set_title('RIFT duration distribution')
    ax.legend()

    # Duration over session (check for drift)
    ax = axes[1]
    ax.plot(rift_durations_s * 1000, '.', markersize=3, alpha=0.6, color='steelblue')
    ax.axhline(NOMINAL_RIFT_DUR * 1000, color='red', linestyle='--', linewidth=1.5, label='Nominal')
    ax.set_xlabel('Trial number')
    ax.set_ylabel('Duration (ms)')
    ax.set_title('RIFT duration over session')
    ax.legend()

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"{s['subject_id']}_rift_duration_check.png"), dpi=150)
    plt.show()
    print(f"  Figure saved → {s['subject_id']}_rift_duration_check.png")

In [ ]:
# Cell 9 - Print inter-trigger interval summaries for key event pairs

for s in subjects:
    raw    = raw_dict[s['subject_id']]
    events = events_dict[s['subject_id']]
    sfreq  = raw.info['sfreq']

    pairs = [
        (TRIG_TRIAL_START, TRIG_MEMORY_ON,  '20→30  ITI          (expect 0.5–0.8 s)'),
        (TRIG_MEMORY_ON,   TRIG_RIFT_ON,    '30→40  Memory item  (expect ~0.25 s)  '),
        (TRIG_RIFT_ON,     TRIG_RIFT_OFF,   '40→50  RIFT window  (expect ~2.00 s)  '),
    ]

    print(f"\n{s['subject_id']} — inter-trigger interval summary:")
    print('-' * 55)
    for code_a, code_b, label in pairs:
        times_a = events[events[:, 2] == code_a, 0] / sfreq
        times_b = events[events[:, 2] == code_b, 0] / sfreq
        n       = min(len(times_a), len(times_b))
        diffs   = (times_b[:n] - times_a[:n])
        diffs   = diffs[diffs > 0]
        print(f'  {label}')
        print(f'    mean={np.mean(diffs)*1000:.1f} ms  std={np.std(diffs)*1000:.1f} ms  '
              f'min={np.min(diffs)*1000:.1f} ms  max={np.max(diffs)*1000:.1f} ms\n')

In [ ]:
# Cell 10 - Load data & inspect PSD

for s in subjects:
    raw = raw_dict[s['subject_id']]
    raw.load_data()

    print(f"\n{s['subject_id']} — data loaded")
    print(f"  Shape: {raw._data.shape}  (channels × samples)")

    fig = raw.compute_psd(fmin=1, fmax=120, picks='eeg').plot(average=True, show=False)
    fig.suptitle(f"{s['subject_id']} — Raw PSD (EEG channels)", fontsize=12)
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"{s['subject_id']}_psd_raw.png"), dpi=150)
    plt.show()
    print(f"  PSD figure saved → {s['subject_id']}_psd_raw.png")

In [ ]:
# Cell 11 - Save events & frequency info

for s in subjects:
    if s['subject_id'] not in freq_info_all:
        print(f"  ⚠ Skipping {s['subject_id']} — no frequency info (check Cell 7)")
        continue

    info = freq_info_all[s['subject_id']]

    # Save events array
    events_path = os.path.join(OUTPUT_DIR, f"{s['subject_id']}_events.npy")
    np.save(events_path, events_dict[s['subject_id']])

    # Save frequency info (drop the raw durations array before saving to CSV)
    freq_df = pd.DataFrame([{k: v for k, v in info.items() if k != 'rift_durations_s'}])
    freq_path = os.path.join(OUTPUT_DIR, f"{s['subject_id']}_flicker_freq.csv")
    freq_df.to_csv(freq_path, index=False)

    print(f"{s['subject_id']}:")
    print(f"  ✓ Events saved      → {s['subject_id']}_events.npy")
    print(f"  ✓ Freq info saved   → {s['subject_id']}_flicker_freq.csv")

print(f'\n{"="*55}')
print(f'  NOTEBOOK 01 COMPLETE')
print(f'  Subjects processed: {len(freq_info_all)}')
for sub_id, info in freq_info_all.items():
    print(f'  {sub_id} — empirical flicker: {info["empirical_flicker_hz"]:.4f} Hz  |  trials: {info["n_rift_trials"]}')
print(f'{"="*55}')
print('→ Continue to Notebook 02: Preprocessing')


